# Data Engineering Lifecycle

### Data Generation

In [1]:
import numpy as np

sensor_temp_data = np.random.uniform(15, 35, (10, 5)) # range values(from 15 to 35) and 10 rows and 5 columns
sensor_temp_data

array([[23.20031498, 23.5895362 , 32.18451651, 31.00931193, 24.06215895],
       [25.33011924, 23.88515951, 20.19002886, 25.51954554, 16.77066383],
       [27.41329569, 18.23218581, 27.80620167, 20.24758816, 17.85499478],
       [27.08372476, 32.45726926, 24.8918759 , 33.32239399, 15.33150763],
       [32.38499484, 33.87374747, 31.88758924, 25.4668734 , 17.62337363],
       [15.84581177, 22.10174731, 24.03721011, 29.52485417, 23.53323556],
       [32.77217943, 17.83235785, 27.66058699, 28.40041469, 24.47965637],
       [33.77713448, 18.77698663, 19.08071332, 31.27935564, 27.50140942],
       [20.3602336 , 26.3781697 , 28.54492272, 20.96224877, 34.33269053],
       [16.54918214, 29.67696007, 18.70003315, 26.98643947, 15.89257607]])

### Data Storage

In [2]:
import pandas as pd
df = [
    {"morning": [23, 25, 22, 20, 24], 
    #  "morning" : np.random.uniform(15, 35, 5).tolist(),
    #  "afternoon": np.random.uniform(15, 35, 5).tolist(),
    #  "evening": np.random.uniform(15, 35, 5).tolist()
     "afternoon": [30, 28, 27, 29, 31], 
     "evening": [20, 18, 19, 21, 22]
     }
]
df1 = pd.DataFrame(df)
df1.to_csv("temp_data.csv", index=False)

### Data Ingestion

In [3]:
pd.read_csv("temp_data.csv").head()

,morning,afternoon,evening
0,"[23, 25, 22, 20, 24]","[30, 28, 27, 29, 31]","[20, 18, 19, 21, 22]"


### DATA TRANSFORMATION

In [4]:
df['avg_temp'] = []

TypeError: list indices must be integers or slices, not str

### DATA SERVING

In [ ]:
df1.to_csv("temp_data_cleaned.csv", index=False)

### DATA PIPELINE

In [ ]:
# Step 1: Data Generation

students = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "David", "Eve"],
    "scores": [85, 92, 781, 90, np.nan]
})

# Step 2: Data Storage
students.to_csv("students.csv", index=False)

# Step 3: Data Ingestion - Reading into memory
students_df = pd.read_csv("students.csv")
students_df

# Step 4: Transformation
students_df['scores'] = students_df['scores'].replace(781, 100)  # Replace the outlier score with a valid score
students_df['scores'] = students_df['scores'].fillna(students_df['scores'].mode()[0])  # Fill NaN values with the mode score
print(students_df)

# Step 5: Data Serving
students_df.to_csv("students_cleaned.csv", index=False)


      name  scores
0    Alice    85.0
1      Bob    92.0
2  Charlie   100.0
3    David    90.0
4      Eve    85.0


# DATA ENGINEERING LIFECYCLE WITH SQL

In [6]:
import sqlite3

conn = sqlite3.connect('students.db')

In [ ]:
#conn.execute("DROP TABLE IF EXISTS students_info;")

In [20]:
create_tables = """
CREATE TABLE IF NOT EXISTS students_info(
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    phone TEXT ,
    email TEXT UNIQUE NOT NULL,
    course TEXT,
    profile_pic TEXT
);  
"""
conn.execute(create_tables)

In [21]:
inserting_data = """
INSERT OR REPLACE INTO students_info (name, phone, email, course, profile_pic) VALUES
('Alice', '123-456-7890', 'alice@example.com', 'Mathematics', 'profile_pic_alice.jpg'),
('Bob', NULL, 'bob@example.com', 'Physics', 'profile_pic_bob.jpg'),
('Charlie', '555-555-5555', 'charlie@example.com', NULL, 'profile_pic_charlie.jpg')
;
"""
conn.execute(inserting_data)

In [22]:
conn.execute("SELECT * FROM students_info").fetchall()

[(1,
  'Alice',
  '123-456-7890',
  'alice@example.com',
  'Mathematics',
  'profile_pic_alice.jpg'),
 (2, 'Bob', None, 'bob@example.com', 'Physics', 'profile_pic_bob.jpg'),
 (3,
  'Charlie',
  '555-555-5555',
  'charlie@example.com',
  None,
  'profile_pic_charlie.jpg')]

In [23]:
pd.read_sql_query("SELECT * FROM students_info", conn)

,student_id,name,phone,email,course,profile_pic
0,1,Alice,123-456-7890,alice@example.com,Mathematics,profile_pic_alice.jpg
1,2,Bob,None,bob@example.com,Physics,profile_pic_bob.jpg
2,3,Charlie,555-555-5555,charlie@example.com,None,profile_pic_charlie.jpg


In [25]:
transformation = """
DELETE FROM students_info
WHERE COURSE IS NULL;

UPDATE students_info
SET phone = '0756890911'
WHERE phone IS NULL ;

UPDATE students_info
SET profile_pic = 'default.jpg'
WHERE profile_pic IS NULL;
"""

conn.executescript(transformation)

In [26]:
pd.read_sql_query("SELECT * FROM students_info", conn)

,student_id,name,phone,email,course,profile_pic
0,1,Alice,123-456-7890,alice@example.com,Mathematics,profile_pic_alice.jpg
1,2,Bob,0756890911,bob@example.com,Physics,profile_pic_bob.jpg


In [27]:
pd.read_sql("SELECT COUNT(*) AS course_count FROM students_info", conn)

,course_count
0,2


In [28]:
inserting_data = """
INSERT OR REPLACE INTO students_info (name, phone, email, course, profile_pic) VALUES
('Dae', '123-456-7890', 'dae@example.com', 'Mathematics', 'profile_pic_dae.jpg')
;
"""
conn.execute(inserting_data)

In [29]:
pd.read_sql_query("SELECT * FROM students_info", conn)

,student_id,name,phone,email,course,profile_pic
0,1,Alice,123-456-7890,alice@example.com,Mathematics,profile_pic_alice.jpg
1,2,Bob,0756890911,bob@example.com,Physics,profile_pic_bob.jpg
2,4,Dae,123-456-7890,dae@example.com,Mathematics,profile_pic_dae.jpg


In [33]:
students_in_math = pd.read_sql_query("""
                                     SELECT * FROM students_info 
                                     WHERE course = 'Mathematics' 
                                     ORDER BY name DESC""", conn)
students_in_math

,student_id,name,phone,email,course,profile_pic
0,4,Dae,123-456-7890,dae@example.com,Mathematics,profile_pic_dae.jpg
1,1,Alice,123-456-7890,alice@example.com,Mathematics,profile_pic_alice.jpg
